In [1]:
import os
import re
import glob
import shlex
import subprocess
from datetime import datetime

from tqdm import tqdm
from PIL import Image


# ============================================================
# ############### CORE IMPORTABLE FUNCTION ###################
# ============================================================
def _img2vid_2012_i3_GET_mp4_water_ultrasmooth_autofallback(
    folder_in,
    path_out=None,
    ext_list=("png", "jpg", "jpeg"),

    # timings
    fps_out=60,
    hold_seconds=2.0,
    trans_seconds=1.25,

    # base encode
    codec="libx264",
    crf=17,
    preset="slow",
    pix_fmt="yuv420p",

    # size handling
    pad_to_first=True,
    force_even_dims=True,

    # motion
    motion_zoom_start=1.00,
    motion_zoom_end=1.06,
    motion_pan_mode="center",     # "center" or "drift"
    motion_drift_px=80,

    # transition style
    xfade_type="smoothleft",      # nice + soft; other: "fade", "smoothup", "circleopen", etc.

    # water (best path uses displace)
    water_strength=12,            # displace intensity
    water_speed=0.22,
    water_blur=2.0,
    water_mix=0.35,

    # water (fallback path without displace)
    fallback_noise_strength=0.22, # 0.10-0.35
    fallback_noise_scale=22,      # noise granularity
    fallback_softlight_opacity=0.28,
    fallback_blur=0.9,

    # extra smooth
    want_minterpolate=True,       # if filter exists, use it; otherwise fallback to fps
    mi_mode="mci",
    mi_mc_mode="aobmc",
    mi_me_mode="bidir",
    mi_vs=1,

    # optional audio
    audio_path=None,
    audio_gain_db=0.0,
    audio_fade=True,

    # debug / logs
    verbose=True,
    keep_debug_cmd=False
):
    """
    Ultra-smooth slideshow with watery transitions.
    Auto-detects ffmpeg filter support and falls back gracefully.
    """

    # ---------------------------
    # helpers
    # ---------------------------
    def _run_capture(cmd_list):
        return subprocess.run(
            cmd_list,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

    def _ffmpeg_exists():
        r = _run_capture(["ffmpeg", "-version"])
        return r.returncode == 0

    def _get_ffmpeg_filters_set():
        r = _run_capture(["ffmpeg", "-hide_banner", "-filters"])
        if r.returncode != 0:
            return set()
        # parse: each filter line includes " ... T.C displace ... "
        # we'll just do a cheap "word in output" check later too
        lines = r.stdout.splitlines()
        found = set()
        for ln in lines:
            # filter name is usually last token after flags
            # but easiest: regex grab " <name> " near end
            m = re.search(r"\s([a-zA-Z0-9_]+)\s*$", ln.strip())
            if m:
                found.add(m.group(1))
        return found

    def _nat_key(p):
        b = os.path.basename(p)
        return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", b)]

    # ---------------------------
    # validate
    # ---------------------------
    if not os.path.isdir(folder_in):
        raise FileNotFoundError(f"Folder not found: {folder_in}")

    if not _ffmpeg_exists():
        raise RuntimeError(
            "ffmpeg not found.\n"
            "Install on mac:\n"
            "  brew install ffmpeg"
        )

    # ---------------------------
    # collect images
    # ---------------------------
    paths = []
    for ext in ext_list:
        paths += glob.glob(os.path.join(folder_in, f"*.{ext}"))
        paths += glob.glob(os.path.join(folder_in, f"*.{ext.upper()}"))

    paths = [p for p in paths if not os.path.basename(p).startswith("._") and not os.path.basename(p).startswith(".DS")]
    if len(paths) < 2:
        raise ValueError(f"Need at least 2 images. Found {len(paths)} in: {folder_in}")

    paths = sorted(paths, key=_nat_key)

    # ---------------------------
    # output path
    # ---------------------------
    if path_out is None:
        stamp = datetime.now().strftime("%m_%d_%H%M%S")
        base = os.path.basename(os.path.normpath(folder_in))
        path_out = os.path.join(folder_in, f"__out_{stamp}_{base}_waterUltra.mp4")
    path_out = os.path.abspath(path_out)
    os.makedirs(os.path.dirname(path_out), exist_ok=True)

    # ---------------------------
    # base dims
    # ---------------------------
    with Image.open(paths[0]) as im:
        w0, h0 = im.size

    if pad_to_first:
        if force_even_dims:
            w0 = w0 if (w0 % 2 == 0) else w0 - 1
            h0 = h0 if (h0 % 2 == 0) else h0 - 1

    # ---------------------------
    # durations
    # ---------------------------
    hold_seconds = float(hold_seconds)
    trans_seconds = float(trans_seconds)

    if trans_seconds >= hold_seconds:
        trans_seconds = max(0.25, hold_seconds * 0.5)

    total_seconds = (len(paths) * hold_seconds) - ((len(paths) - 1) * trans_seconds)

    # ---------------------------
    # detect filter support
    # ---------------------------
    filtset = _get_ffmpeg_filters_set()

    has_displace = ("displace" in filtset)
    has_minterpolate = ("minterpolate" in filtset)

    # If parse failed, do robust fallback check:
    # (some builds still list; but if parsing misses, do substring check)
    if not has_displace or not has_minterpolate:
        r2 = _run_capture(["ffmpeg", "-hide_banner", "-filters"])
        if r2.returncode == 0:
            txt = r2.stdout
            if " displace " in txt:
                has_displace = True
            if " minterpolate " in txt:
                has_minterpolate = True

    use_displace = has_displace
    use_minterpolate = bool(want_minterpolate and has_minterpolate)

    if verbose:
        print("\n--- IMG → MP4 (WATER ULTRA / AUTO) ---")
        print(f"Folder:     {folder_in}")
        print(f"Images:     {len(paths)}")
        print(f"Size base:  {w0}x{h0}")
        print(f"Hold:       {hold_seconds:.2f}s")
        print(f"Trans:      {trans_seconds:.2f}s")
        print(f"Total:      {total_seconds:.2f}s")
        print(f"FPS out:    {fps_out}")
        print(f"ffmpeg displace:      {'YES' if has_displace else 'NO'}")
        print(f"ffmpeg minterpolate:  {'YES' if has_minterpolate else 'NO'}")
        if audio_path:
            print(f"Audio:      {audio_path}")
        print(f"OUT:        {path_out}\n")

    # ---------------------------
    # build ffmpeg inputs
    # ---------------------------
    cmd = ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error"]

    for p in tqdm(paths, desc="Queueing images", ncols=80):
        cmd += ["-loop", "1", "-t", f"{hold_seconds:.6f}", "-i", os.path.abspath(p)]

    if audio_path:
        audio_path = os.path.abspath(audio_path)
        if not os.path.isfile(audio_path):
            raise FileNotFoundError(f"Audio file not found: {audio_path}")
        cmd += ["-i", audio_path]

    # ---------------------------
    # motion expressions
    # ---------------------------
    fps_base = int(fps_out) if int(fps_out) > 0 else 60
    frames_hold = max(2, int(round(hold_seconds * fps_base)))

    if motion_pan_mode not in ("center", "drift"):
        motion_pan_mode = "center"

    if motion_pan_mode == "center":
        x_expr = "(iw-ow)/2"
        y_expr = "(ih-oh)/2"
    else:
        amp = float(motion_drift_px)
        x_expr = f"(iw-ow)/2 + {amp:.2f}*sin(on*0.012)"
        y_expr = f"(ih-oh)/2 + {amp:.2f}*cos(on*0.010)"

    z0 = float(motion_zoom_start)
    z1 = float(motion_zoom_end)
    if z1 < z0:
        z0, z1 = z1, z0
    if z0 < 1.0:
        z0 = 1.0
    if z1 < 1.0:
        z1 = 1.0

    zoom_expr = f"{z0:.5f} + ({z1:.5f}-{z0:.5f})*on/{max(1, frames_hold-1)}"

    if pad_to_first:
        base_geom = (
            f"scale=w={w0}:h={h0}:force_original_aspect_ratio=decrease,"
            f"pad=w={w0}:h={h0}:x=(ow-iw)/2:y=(oh-ih)/2"
        )
    else:
        base_geom = "scale=trunc(iw/2)*2:trunc(ih/2)*2"

    # ---------------------------
    # filter_complex
    # ---------------------------
    fc = []
    vlabels = []

    # per-image normalize + motion
    for i in range(len(paths)):
        chain = (
            f"[{i}:v]"
            f"{base_geom},"
            f"zoompan=z='{zoom_expr}':x='{x_expr}':y='{y_expr}':d={frames_hold}:s={w0}x{h0},"
            f"fps={fps_base},setsar=1"
        )
        if force_even_dims:
            chain += ",scale=trunc(iw/2)*2:trunc(ih/2)*2"
        out = f"[v{i}]"
        fc.append(chain + out)
        vlabels.append(out)

    # chain xfade
    cur = vlabels[0]
    for k in range(1, len(vlabels)):
        offset = (k * hold_seconds) - (k * trans_seconds)
        nxt = vlabels[k]
        out = f"[vx{k}]"
        fc.append(
            f"{cur}{nxt}xfade=transition={xfade_type}:duration={trans_seconds:.6f}:offset={offset:.6f}{out}"
        )
        cur = out

    vxf = cur

    # ---------------------------
    # WATER EFFECT (auto path)
    # ---------------------------
    ws = float(water_strength)
    sp = float(water_speed)
    wb = float(water_blur)
    wm = float(water_mix)

    if use_displace:
        # BEST: true displacement map
        fc.append(
            f"color=c=gray:s={w0}x{h0}:d={total_seconds:.6f},"
            f"noise=alls=20:allf=t+u,"
            f"gblur=sigma={wb:.3f},format=rgba[dmap]"
        )
        fc.append(
            f"[dmap]rotate=0.010*sin(2*PI*t*{sp:.4f}):c=none:ow={w0}:oh={h0},format=rgba[dmap2]"
        )
        fc.append(
            f"{vxf}[dmap2]displace={ws:.3f}:{ws:.3f}[vdisp]"
        )
        fc.append(
            f"{vxf}[vdisp]blend=all_mode=overlay:all_opacity={wm:.3f}[vwater]"
        )
        water_out = "[vwater]"
    else:
        # GOOD fallback: “liquid look” without displace
        # - animated noise layer
        # - softlight blend + a tiny blur
        # - slight chroma shift / shimmer (subtle)
        ns = float(fallback_noise_strength)
        sc = int(fallback_noise_scale)
        op = float(fallback_softlight_opacity)
        bl = float(fallback_blur)

        fc.append(
            f"color=c=gray:s={w0}x{h0}:d={total_seconds:.6f},"
            f"noise=alls={sc}:allf=t+u,"
            f"gblur=sigma={bl:.3f},"
            f"rotate=0.006*sin(2*PI*t*{sp:.4f}):c=none:ow={w0}:oh={h0},"
            f"eq=contrast=1.06:brightness=0.00:saturation=1.00,"
            f"format=rgba[nlayer]"
        )
        # Blend noise into video with softlight (waterish refraction feel)
        fc.append(
            f"{vxf}[nlayer]blend=all_mode=softlight:all_opacity={op:.3f}[vsoft]"
        )
        # Add tiny temporal mix to smooth shimmer
        fc.append(
            f"[vsoft]tmix=frames=3:weights='1 2 1',"
            f"eq=contrast=1.00:brightness=0.00:saturation=1.00,"
            f"gblur=sigma={ns:.3f}[vwater]"
        )
        water_out = "[vwater]"

    # ---------------------------
    # extra smooth output
    # ---------------------------
    if use_minterpolate:
        fc.append(
            f"{water_out}"
            f"minterpolate=fps={int(fps_out)}:mi_mode={mi_mode}:mc_mode={mi_mc_mode}:me_mode={mi_me_mode}:vsbmc={mi_vs}"
            f"[vout]"
        )
    else:
        fc.append(f"{water_out}fps={int(fps_out)}[vout]")

    vout = "[vout]"

    # ---------------------------
    # audio chain
    # ---------------------------
    amap = None
    if audio_path:
        aidx = len(paths)
        achain = f"[{aidx}:a]atrim=0:{total_seconds:.6f},asetpts=PTS-STARTPTS"

        if float(audio_gain_db) != 0.0:
            achain += f",volume={float(audio_gain_db):.3f}dB"

        if audio_fade:
            fade_d = min(1.5, max(0.25, trans_seconds))
            end_start = max(0.0, total_seconds - fade_d)
            achain += f",afade=t=in:st=0:d={fade_d:.3f}"
            achain += f",afade=t=out:st={end_start:.3f}:d={fade_d:.3f}"

        amap = "[aout]"
        fc.append(achain + amap)

    filter_complex = ";".join(fc)

    # ---------------------------
    # finalize cmd
    # ---------------------------
    cmd += ["-filter_complex", filter_complex, "-map", vout]

    if amap:
        cmd += ["-map", amap]
    else:
        cmd += ["-an"]

    cmd += [
        "-c:v", codec,
        "-crf", str(crf),
        "-preset", preset,
        "-pix_fmt", pix_fmt,
        "-movflags", "+faststart",
        path_out
    ]

    # ---------------------------
    # run ffmpeg (capture real error!)
    # ---------------------------
    r = _run_capture(cmd)

    if r.returncode != 0:
        cmd_str = " ".join(shlex.quote(x) for x in cmd)

        msg = "\n".join([
            "ffmpeg failed with this stderr (THIS is the real clue):",
            "----------------------------------------------------",
            (r.stderr.strip() if r.stderr else "(no stderr captured)"),
            "----------------------------------------------------",
            "",
            "Full command used:",
            cmd_str
        ])

        # optionally write cmd to a debug file
        if keep_debug_cmd:
            dbg = os.path.join(folder_in, "__ffmpeg_debug_cmd.txt")
            with open(dbg, "w", encoding="utf-8") as f:
                f.write(cmd_str + "\n\n" + (r.stderr or ""))
            if verbose:
                print(f"\nWrote debug command to: {dbg}\n")

        raise RuntimeError(msg)

    if verbose:
        print("\nDONE ✅")
        print(f"Saved: {path_out}\n")

    return path_out


In [2]:
folder_in = input("\nPaste folder path with PNG/JPGs:\n> ").strip().strip('"').strip("'")

out_mp4 = _img2vid_2012_i3_GET_mp4_water_ultrasmooth_autofallback(
    folder_in=folder_in,
    fps_out=60,
    hold_seconds=2.0,
    trans_seconds=1.25,
    want_minterpolate=True,     # will auto-disable if not supported
    keep_debug_cmd=True,        # drops a __ffmpeg_debug_cmd.txt if it fails
)

print(out_mp4)



Paste folder path with PNG/JPGs:
>  /Users/yerik/Desktop/_12_29_SPKR_ALL_FEMME/VID_material



--- IMG → MP4 (WATER ULTRA / AUTO) ---
Folder:     /Users/yerik/Desktop/_12_29_SPKR_ALL_FEMME/VID_material
Images:     7
Size base:  1616x2000
Hold:       2.00s
Trans:      1.25s
Total:      6.50s
FPS out:    60
ffmpeg displace:      YES
ffmpeg minterpolate:  YES
OUT:        /Users/yerik/Desktop/_12_29_SPKR_ALL_FEMME/VID_material/__out_12_20_161816_VID_material_waterUltra.mp4



Queueing images: 100%|████████████████████████| 7/7 [00:00<00:00, 132252.83it/s]


Wrote debug command to: /Users/yerik/Desktop/_12_29_SPKR_ALL_FEMME/VID_material/__ffmpeg_debug_cmd.txt



RuntimeError: ffmpeg failed with this stderr (THIS is the real clue):
----------------------------------------------------
[AVFilterGraph @ 0x944c28000] No option name near '12.000'
[AVFilterGraph @ 0x944c28000] Error parsing a filter description around: [vdisp];[vx6][vdisp]blend=all_mode=overlay:all_opacity=0.350[vwater];[vwater]minterpolate=fps=60:mi_mode=mci:mc_mode=aobmc:me_mode=bidir:vsbmc=1[vout]
[AVFilterGraph @ 0x944c28000] Error parsing filterchain '[vx6][dmap2]displace=12.000:12.000[vdisp];[vx6][vdisp]blend=all_mode=overlay:all_opacity=0.350[vwater];[vwater]minterpolate=fps=60:mi_mode=mci:mc_mode=aobmc:me_mode=bidir:vsbmc=1[vout]' around: [vdisp];[vx6][vdisp]blend=all_mode=overlay:all_opacity=0.350[vwater];[vwater]minterpolate=fps=60:mi_mode=mci:mc_mode=aobmc:me_mode=bidir:vsbmc=1[vout]
Failed to set value '[0:v]scale=w=1616:h=2000:force_original_aspect_ratio=decrease,pad=w=1616:h=2000:x=(ow-iw)/2:y=(oh-ih)/2,zoompan=z='1.00000 + (1.06000-1.00000)*on/119':x='(iw-ow)/2':y='(ih